In [1]:
import pandas as pd

In [2]:
tfidf_results = pd.read_csv(
    "../data/processed/tfidf_matching_results.csv"
)

skill_results = pd.read_csv(
    "../data/processed/skill_matching_results.csv"
)

experience_results = pd.read_csv(
    "../data/processed/experience_results.csv"
)

semantic_results = pd.read_csv(
    "../data/processed/semantic_matching_results.csv"
)

In [3]:
print("TF-IDF:", tfidf_results.shape)
print("Skills:", skill_results.shape)
print("Experience:", experience_results.shape)
print("Semantic:", semantic_results.shape)

TF-IDF: (2481, 4)
Skills: (2481, 7)
Experience: (2481, 4)
Semantic: (2481, 4)


In [4]:
print(tfidf_results.columns)
print(skill_results.columns)
print(experience_results.columns)
print(semantic_results.columns)

Index(['ID', 'Category', 'match_score', 'rank'], dtype='str')
Index(['skill_rank', 'ID', 'Category', 'extracted_skills', 'matched_skills',
       'missing_skills', 'skill_coverage'],
      dtype='str')
Index(['ID', 'Category', 'calculated_experience', 'experience_score'], dtype='str')
Index(['semantic_rank', 'ID', 'Category', 'semantic_score'], dtype='str')


In [5]:
hybrid_results = tfidf_results[
    ["ID", "Category", "match_score"]
].copy()

In [6]:
hybrid_results = hybrid_results.merge(
    skill_results[
        ["ID", "skill_coverage"]
    ],
    on="ID",
    how="inner"
)

In [7]:
hybrid_results = hybrid_results.merge(
    experience_results[
        ["ID", "experience_score"]
    ],
    on="ID",
    how="inner"
)

In [8]:
hybrid_results = hybrid_results.merge(
    semantic_results[
        ["ID", "semantic_score"]
    ],
    on="ID",
    how="inner"
)

In [9]:
print(hybrid_results.shape)

(2481, 6)


In [10]:
hybrid_results.head()

,ID,Category,match_score,skill_coverage,experience_score,semantic_score
0,62994611,AGRICULTURE,21.273043,71.428571,100.0,42.507850
1,43311839,CONSULTANT,16.315918,42.857143,0.0,36.977123
2,51588273,ENGINEERING,16.231933,28.571429,0.0,30.018085
3,36434348,INFORMATION-TECHNOLOGY,15.815803,14.285714,0.0,19.115088
4,42156237,DIGITAL-MEDIA,14.860788,28.571429,100.0,19.811626


In [11]:
hybrid_results.isna().sum()

ID                  0
Category            0
match_score         0
skill_coverage      0
experience_score    0
semantic_score      0
dtype: int64

In [12]:
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min()) * 100


hybrid_results["tfidf_normalized"] = min_max_normalize(
    hybrid_results["match_score"]
)

In [13]:
hybrid_results["semantic_normalized"] = min_max_normalize(
    hybrid_results["semantic_score"]
)

In [14]:
hybrid_results[
    [
        "ID",
        "match_score",
        "tfidf_normalized",
        "semantic_score",
        "semantic_normalized"
    ]
].head()

,ID,match_score,tfidf_normalized,semantic_score,semantic_normalized
0,62994611,21.273043,100.000000,42.507850,96.178094
1,43311839,16.315918,76.697622,36.977123,85.270137
2,51588273,16.231933,76.302826,30.018085,71.545198
3,36434348,15.815803,74.346688,19.115088,50.041800
4,42156237,14.860788,69.857366,19.811626,51.415545


In [15]:
hybrid_results["hybrid_score"] = (
    0.35 * hybrid_results["skill_coverage"]
    + 0.30 * hybrid_results["semantic_normalized"]
    + 0.20 * hybrid_results["tfidf_normalized"]
    + 0.15 * hybrid_results["experience_score"]
)

In [16]:
hybrid_results = hybrid_results.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(drop=True)

In [17]:
hybrid_results[
    [
        "hybrid_rank",
        "ID",
        "Category",
        "tfidf_normalized",
        "skill_coverage",
        "experience_score",
        "semantic_normalized",
        "hybrid_score"
    ]
].head(10)

KeyError: "['hybrid_rank'] not in index"

In [18]:
hybrid_results["hybrid_score"] = (
    0.35 * hybrid_results["skill_coverage"]
    + 0.30 * hybrid_results["semantic_normalized"]
    + 0.20 * hybrid_results["tfidf_normalized"]
    + 0.15 * hybrid_results["experience_score"]
)

In [19]:
hybrid_results = hybrid_results.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(drop=True)

hybrid_results["hybrid_rank"] = range(
    1,
    len(hybrid_results) + 1
)

In [20]:
hybrid_results[
    [
        "hybrid_rank",
        "ID",
        "Category",
        "tfidf_normalized",
        "skill_coverage",
        "experience_score",
        "semantic_normalized",
        "hybrid_score"
    ]
].head(10)

,hybrid_rank,ID,Category,tfidf_normalized,skill_coverage,experience_score,semantic_normalized,hybrid_score
0,1,62994611,AGRICULTURE,100.000000,71.428571,100.0,96.178094,88.853428
1,2,83816738,INFORMATION-TECHNOLOGY,63.426373,42.857143,100.0,100.000000,72.685275
2,3,12011623,ENGINEERING,54.453264,28.571429,100.0,89.045326,62.604251
3,4,27152464,ENGINEERING,59.692042,28.571429,100.0,73.688723,59.045025
4,5,95429627,CONSULTANT,36.373574,42.857143,100.0,67.398948,57.494399
5,6,11981094,ENGINEERING,57.422693,28.571429,100.0,66.062033,56.303149
6,7,43311839,CONSULTANT,76.697622,42.857143,0.0,85.270137,55.920565
7,8,39441617,CONSULTANT,41.651390,28.571429,100.0,73.015368,55.234888
8,9,30863060,CONSULTANT,51.202059,28.571429,100.0,65.225229,54.807980
9,10,42156237,DIGITAL-MEDIA,69.857366,28.571429,100.0,51.415545,54.396137


In [21]:
print("Minimum hybrid score:", hybrid_results["hybrid_score"].min())
print("Maximum hybrid score:", hybrid_results["hybrid_score"].max())

Minimum hybrid score: 0.047981768206333736
Maximum hybrid score: 88.85342816799258


In [22]:
hybrid_results.isna().sum()

ID                     0
Category               0
match_score            0
skill_coverage         0
experience_score       0
semantic_score         0
tfidf_normalized       0
semantic_normalized    0
hybrid_score           0
hybrid_rank            0
dtype: int64

In [23]:
hybrid_results.columns

Index(['ID', 'Category', 'match_score', 'skill_coverage', 'experience_score',
       'semantic_score', 'tfidf_normalized', 'semantic_normalized',
       'hybrid_score', 'hybrid_rank'],
      dtype='str')

In [24]:
hybrid_results.to_csv(
    "../data/processed/hybrid_matching_results.csv",
    index=False
)

In [25]:
import os

print(
    os.path.exists(
        "../data/processed/hybrid_matching_results.csv"
    )
)

True
